# 01 · Clean the LCA file (FY2026 Q3)

**Goal:** turn the raw DOL LCA disclosure file (~437k filings, 98 columns) into one tidy table of
**data, ML and analytics roles** that we load into PostgreSQL and Power BI.

All the logic lives in `src/clean_lca.py`. This notebook runs it **one step at a time** and checks
the result after each step, so every choice can be explained.

> **Source:** U.S. Department of Labor, Office of Foreign Labor Certification (OFLC), LCA Programs disclosure data,
> FY2026 Q3 file (covers decisions from Oct 1, 2025 to Jun 30, 2026).
> An LCA is an employer's filing with DOL, **not** an approved H-1B petition. Counts below are LCA filings.
> Informational only, not legal advice.

**Before running:** put `LCA_Disclosure_Data_FY2026_Q3.xlsx` in `data/raw/` (it is git-ignored),
and select the repo's `.venv` kernel.

In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
import clean_lca as cl

RAW_XLSX = REPO / "data" / "raw" / "LCA_Disclosure_Data_FY2026_Q3.xlsx"
OUT_DIR = REPO / "data" / "processed"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
print("pandas", pd.__version__)

pandas 3.1.0


## Step 1 · Load the raw file

`load_quarter()` reads the Excel file with the **calamine** engine (written in Rust, much faster than openpyxl)
and saves a Parquet copy next to it. The first run takes a few minutes; after that it loads the Parquet copy in seconds.

Every column is read **as text** (`dtype=str`). That way pandas never guesses a type (for example turning
ZIP codes into numbers and dropping leading zeros). We set types on purpose in later steps.

In [2]:
raw = cl.load_quarter(RAW_XLSX)
print(raw.shape)
print(raw.attrs)

(437496, 98)
{'source_file': 'LCA_Disclosure_Data_FY2026_Q3.xlsx', 'source_fy': 2026, 'source_quarter': 3}


In [3]:
steps = {}  # row counts after each step, for the summary at the end
steps["1. raw rows"] = len(raw)
assert raw["CASE_NUMBER"].is_unique, "CASE_NUMBER should be unique within one file"
raw["CASE_STATUS"].value_counts()

CASE_STATUS
Certified                401412
Certified - Withdrawn     26303
Withdrawn                  7224
Denied                     2557
Name: count, dtype: int64

## Step 2 · Keep only the columns we need

We keep 33 of the 98 columns: case, status and dates, employer, job and SOC code, worksite, wages, and the
H-1B dependent / willful violator flags.

We **drop** contact-person, attorney and preparer names, emails and phone numbers. We don't need them, and this
project is public.

Column names change a little between DOL releases. This file says `H_1B_DEPENDENT`, while the PDF layout
says `H-1B_DEPENDENT`; `select_columns()` maps both to `H_1B_DEPENDENT`. Empty strings become missing values.

In [4]:
df = cl.select_columns(raw)
print(df.shape, "| columns missing from this file:", df.attrs["missing_columns"])
steps["2. after column selection"] = len(df)

(437496, 33) | columns missing from this file: []


## Step 3 · Status, visa class and fiscal year

* **Dates** are parsed into real dates.
* **Fiscal year / quarter** come from `DECISION_DATE`, using the federal fiscal year (Oct 1 to Sep 30), so an
  October 2025 decision is FY2026 Q1. We also store which file the row came from (`source_fy`, `source_quarter`).
* **`is_certified_h1b`** = status is exactly `Certified` and visa class is `H-1B`. "Certified - Withdrawn" means the
  employer withdrew it after certification, so it is **not** counted. E-3 (Australia) and H-1B1 (Chile, Singapore)
  rows stay in the data with `is_h1b = False`, so you can filter them in SQL.

In [5]:
df = cl.add_status_and_dates(df)
steps["3. after status/dates"] = len(df)
display(pd.crosstab(df["CASE_STATUS"], df["VISA_CLASS"], margins=True))
print("certified H-1B rows:", int(df["is_certified_h1b"].sum()))
print("decision date missing:", int(df["DECISION_DATE"].isna().sum()))
df.groupby(["decision_fy", "decision_fq"]).size().rename("rows")

VISA_CLASS,E-3 Australian,H-1B,H-1B1 Chile,H-1B1 Singapore,All
CASE_STATUS,,,,,
Certified,6702,392175,1485,1050,401412
Certified - Withdrawn,556,25546,92,109,26303
Denied,247,2235,43,32,2557
Withdrawn,166,6996,34,28,7224
All,7671,426952,1654,1219,437496


certified H-1B rows: 392175
decision date missing: 0


decision_fy  decision_fq
2026         1               82950
             2              122739
             3              231807
Name: rows, dtype: int64

## Step 4 · Normalize SOC codes

The SOC code is the government's occupation code (e.g. `15-2051.00` = Data Scientists). Most rows look like
`NN-NNNN.NN`, but some are short (`15-2051`) or messy (`15-1212 .00`, `17-2199-05`, `15-1252.00 - Software Developers`).

`normalize_soc()` creates:
* `soc_6`: the 6-digit code (`15-2051`)
* `soc_8`: the detailed code (`15-2051.01`); a missing suffix becomes `.00`
* `soc_valid`: False when the value isn't a SOC code at all (for example a NAICS industry code like `541618`)

In [6]:
df = cl.normalize_soc(df)
steps["4. after SOC normalization"] = len(df)
print("invalid SOC codes:", int((~df["soc_valid"]).sum()))
display(df.loc[~df["soc_valid"], ["SOC_CODE", "SOC_TITLE"]])
# what the messy codes became
odd = df["SOC_CODE"].str.fullmatch(r"\d{2}-\d{4}\.\d{2}") == False
df.loc[odd, ["SOC_CODE", "soc_8"]].drop_duplicates().head(15)

invalid SOC codes: 1


,SOC_CODE,SOC_TITLE
223935,541618,Other Management Consulting Services


,SOC_CODE,soc_8
101,19-4021,19-4021.00
411,15-1252,15-1252.00
423,25-9031,25-9031.00
434,15-1253,15-1253.00
436,17-2071,17-2071.00
662,15-1243,15-1243.00
1448,29-1223,29-1223.00
1472,15-1211,15-1211.00
1662,15-2041,15-2041.00
1672,13-2051,13-2051.00


## Step 5 · Annualize wages

Employers state pay per year, hour, week, two weeks or month. To compare them we convert everything to **annual dollars**:

| unit | multiply by |
|---|---|
| Year | 1 |
| Hour | 2,080 (40 h × 52 weeks) |
| Week | 52 |
| Bi-Weekly | 26 |
| Month | 12 |

* `wage_annual_from` is the **lower end** of the offered wage. It's the conservative number and the one the dashboard uses.
* `wage_annual_to` is kept when the filing gives a range.
* `pw_annual` is the prevailing (minimum required) wage, annualized the same way. `wage_vs_pw_pct` = how far the offer is above it.
* **Unit typos:** some filings say `Hour` or `Week` but the number is clearly a salary (e.g. 130,000 "per week").
  A non-yearly rate of **15,000 or more** is treated as yearly and flagged `unit_assumed_year`.
* Nothing is deleted. Odd values are **flagged** in `wage_flag` (`below_20k`, `above_1m`) so you can exclude them in SQL
  or Power BI. Use medians, not averages, in the dashboard.
* `PW_WAGE_LEVEL` is blank on about 10% of rows (usually when a survey other than OES was used). Blanks become `Not reported`.

In [7]:
df = cl.annualize_wages(df)
steps["5. after wage annualization"] = len(df)
display(df["WAGE_UNIT_OF_PAY"].value_counts(dropna=False))
display(df["wage_flag"].value_counts())
display(df["PW_WAGE_LEVEL"].value_counts())

WAGE_UNIT_OF_PAY
Year         403531
Hour          32839
Month           741
Week            206
Bi-Weekly       179
Name: count, dtype: int64[pyarrow]

wage_flag
ok                   436818
unit_assumed_year       604
above_1m                 55
below_20k                19
Name: count, dtype: int64[pyarrow]

PW_WAGE_LEVEL
II              156789
III              80756
IV               78211
I                76884
Not reported     44856
Name: count, dtype: int64[pyarrow]

In [8]:
# Wage distribution (all rows, flagged rows removed), in annual dollars
ok = df["wage_flag"].isin(["ok", "unit_assumed_year"])
df.loc[ok, ["wage_annual_from", "pw_annual"]].describe(percentiles=[.05, .25, .5, .75, .95]).round(0)

,wage_annual_from,pw_annual
count,437422.0,437422.0
mean,136981.0,117826.0
std,61720.0,44301.0
min,20000.0,15080.0
5%,66667.0,57658.0
25%,95035.0,86382.0
50%,125278.0,112029.0
75%,164000.0,141814.0
95%,240000.0,201448.0
max,1000000.0,841845.0


In [9]:
# Examples of unit typos that were corrected, and of flagged extremes
cols = ["JOB_TITLE", "WAGE_RATE_OF_PAY_FROM", "WAGE_UNIT_OF_PAY", "PREVAILING_WAGE", "PW_UNIT_OF_PAY", "wage_annual_from", "wage_flag"]
display(df.loc[df["wage_flag"] == "unit_assumed_year", cols].head(8))
display(df.loc[df["wage_flag"].isin(["above_1m", "below_20k"]), cols].head(8))

,JOB_TITLE,WAGE_RATE_OF_PAY_FROM,WAGE_UNIT_OF_PAY,PREVAILING_WAGE,PW_UNIT_OF_PAY,wage_annual_from,wage_flag
531,Senior Software Engineer,130000,Week,129667,Year,130000.0,unit_assumed_year
1504,PROJECT MANAGER,85000,Week,59405,Year,85000.0,unit_assumed_year
1826,"Mid-Level Associate (Lawyer), PCTG (PET)",390000,Bi-Weekly,212826,Year,390000.0,unit_assumed_year
1952,Senior Psychiatrist (Supervisor),30307,Month,20196.76,Month,30307.0,unit_assumed_year
2198,"Director, Technical Management - Software Engineering",157248,Bi-Weekly,157248,Year,157248.0,unit_assumed_year
2205,"Manager, Engineering",173393,Hour,173393,Week,173393.0,unit_assumed_year
2587,Lead Project Engineer,124000,Week,123635,Year,124000.0,unit_assumed_year
2609,SENIOR DATA ENGINEER,178937,Hour,143250,Year,178937.0,unit_assumed_year


,JOB_TITLE,WAGE_RATE_OF_PAY_FROM,WAGE_UNIT_OF_PAY,PREVAILING_WAGE,PW_UNIT_OF_PAY,wage_annual_from,wage_flag
14655,Research Engineer,2000000,Year,164403,Year,2000000.0,above_1m
27659,Executive Producer,17500,Month,206856,Year,17500.0,below_20k
35778,Account Director,15417,Month,14443.84,Month,15417.0,below_20k
38722,Physician (Interventional Cardiologist/Medical Director),1200000,Year,588453,Year,1200000.0,above_1m
51124,Director,1295000,Year,109283,Year,1295000.0,above_1m
63611,Project Executive II,15816.67,Month,10316.78,Month,15817.0,below_20k
66362,Lead SQL Reports Developer,15567084,Year,125209,Year,15567084.0,above_1m
82931,Developer,8,Hour,8,Hour,16640.0,below_20k


## Step 6 · Standardize employer names

The same company is spelled many ways (`Google LLC`, `GOOGLE, L.L.C.`). Two layers:

1. **`employer_name_clean`**: uppercase, `&` becomes `AND`, punctuation is removed, a leading `THE` is dropped, and
   legal-form words at the end are dropped (INC, LLC, CORP, CORPORATION, CO, LTD, LP, LLP, PLLC, NA ...), including the `AND` left over from `& Co.`.
2. **`employer_name_std`**: rows are grouped by **FEIN** (the employer's federal tax ID). Each FEIN gets the clean
   name it is filed under most often. This fixes spelling variants of the same legal entity.

FEIN identifies a **legal entity, not a brand**. Amazon, for example, files under several entities with different
FEINs, and they stay separate. Merging those into one brand needs a small manual mapping table, which is a later improvement.

In [10]:
df = cl.standardize_employers(df)
steps["6. after employer standardization"] = len(df)
print("distinct raw names:  ", df["EMPLOYER_NAME"].nunique())
print("distinct clean names:", df["employer_name_clean"].nunique())
print("distinct std names:  ", df["employer_name_std"].nunique())
print("distinct valid FEINs:", df.loc[df["fein_valid"], "EMPLOYER_FEIN"].nunique())
print("rows with invalid FEIN:", int((~df["fein_valid"]).sum()))

distinct raw names:   61117
distinct clean names: 56258
distinct std names:   54868
distinct valid FEINs: 55238
rows with invalid FEIN: 0


In [11]:
# Spot check: the raw spellings that were merged into a few big employers
for name in ["GOOGLE", "MICROSOFT", "AMAZON COM SERVICES", "META PLATFORMS"]:
    variants = df.loc[df["employer_name_std"] == name, "EMPLOYER_NAME"].value_counts()
    print(f"\n{name}: {variants.sum():,} rows, {len(variants)} raw spellings")
    print(variants.head(6).to_string())


GOOGLE: 7,486 rows, 1 raw spellings
EMPLOYER_NAME
Google LLC    7486

MICROSOFT: 5,159 rows, 1 raw spellings
EMPLOYER_NAME
Microsoft Corporation    5159

AMAZON COM SERVICES: 13,931 rows, 3 raw spellings
EMPLOYER_NAME
Amazon.com Services LLC    13686
AMAZON.COM SERVICES LLC      244
Amazon Services LLC            1

META PLATFORMS: 5,281 rows, 3 raw spellings
EMPLOYER_NAME
Meta Platforms, Inc    5279
Newton, Inc.              1
Polmer LLC                1


## Step 7 · Normalize job titles

* `job_title_clean`: uppercase, punctuation becomes spaces, abbreviations expanded from the `TITLE_ABBREVIATIONS` constant (`BUSINESS INTEL` → `BUSINESS INTELLIGENCE`, `SR` → `SENIOR`, `ENGR`/`ENG`/`ENGG` → `ENGINEER`, `MGR` → `MANAGER`, `ANLST` → `ANALYST`, `SCI` → `SCIENTIST`, `DEV` → `DEVELOPER` but `BUS DEV` → `BUSINESS DEVELOPMENT`, `OPERATION RESEARCH` → `OPERATIONS RESEARCH` ...). Without this, "Business Intel Engineer" and "Business Intelligence Engineer" would count as two titles.
* `job_title_base`: `job_title_clean` with seniority and level words removed (SENIOR, LEAD, PRINCIPAL, STAFF, II, 3, L5 ...)
  and text in brackets dropped. That groups `Senior Data Scientist II` with `Data Scientist`, which the
  **"Top sponsors for your title"** page needs.

In [12]:
df = cl.normalize_titles(df)
steps["7. after title normalization"] = len(df)
print("distinct raw titles:  ", df["JOB_TITLE"].nunique())
print("distinct clean titles:", df["job_title_clean"].nunique())
print("distinct base titles: ", df["job_title_base"].nunique())
df[["JOB_TITLE", "job_title_clean", "job_title_base"]].sample(10, random_state=42)

distinct raw titles:   99781
distinct clean titles: 89060
distinct base titles:  70893


,JOB_TITLE,job_title_clean,job_title_base
427608,SUPPLY CHAIN ANALYST II,SUPPLY CHAIN ANALYST II,SUPPLY CHAIN ANALYST
121419,Software Development Engineer,SOFTWARE DEVELOPMENT ENGINEER,SOFTWARE DEVELOPMENT ENGINEER
113780,Senior Software Engineer,SENIOR SOFTWARE ENGINEER,SOFTWARE ENGINEER
148190,Senior Payments & Fraud Analyst,SENIOR PAYMENTS AND FRAUD ANALYST,PAYMENTS AND FRAUD ANALYST
267273,Principal Data Engineer,PRINCIPAL DATA ENGINEER,DATA ENGINEER
136849,Hardware Development Engineer II,HARDWARE DEVELOPMENT ENGINEER II,HARDWARE DEVELOPMENT ENGINEER
175702,Software Developer,SOFTWARE DEVELOPER,SOFTWARE DEVELOPER
405893,Software Engineering,SOFTWARE ENGINEERING,SOFTWARE ENGINEERING
207902,BUSINESS ANALYST 3,BUSINESS ANALYST 3,BUSINESS ANALYST
205820,Strategic Partnerships Development Manager,STRATEGIC PARTNERSHIPS DEVELOPMENT MANAGER,STRATEGIC PARTNERSHIPS DEVELOPMENT MANAGER


## Step 8 · Classify role family

This is the most important judgment in the project. Each row gets one `role_family`:
**Data Scientist, ML/AI Engineer, Data Engineer, Statistician/Quant, Data Analyst/BI**, or **Other** (not a data role).

**Rule 1: core SOC codes are always data roles.** The family comes from the title if a title rule matches,
otherwise from the code:

| SOC | official title | default family |
|---|---|---|
| 15-2051.00 | Data Scientists | Data Scientist |
| 15-2051.01 | Business Intelligence Analysts | Data Analyst/BI |
| 15-2041.00 / .01 | Statisticians / Biostatisticians | Statistician/Quant |
| 15-2031.00 | Operations Research Analysts | Statistician/Quant |
| 13-2099.01 | Financial Quantitative Analysts | Statistician/Quant |
| 15-1243.01 | Data Warehousing Specialists | Data Engineer |

**Rule 2: broad SOC codes count only if the title says so.** In 15-1252 (Software Developers) and the other
computer codes (15-1xxx), the remaining math codes (15-2xxx), computer/IS managers (11-3021), market research (13-1161),
management, financial and other business analysts, and economists, a row counts only when its **title** matches a
keyword rule. Many Data Engineers and ML Engineers are filed as Software Developers, so the SOC code alone would miss them.

**Rule 3: everything else is Other.** Titles aren't used outside those codes. That avoids pulling in, for example,
analytical chemists or "AI hardware" roles filed under engineering codes.

**Title rules** are checked in this order, and the first match wins:
1. Data Scientist: `DATA SCIEN`, `DECISION SCIEN`
2. ML/AI Engineer: `MACHINE LEARNING`, `ML`, `MLOPS`, `AI` (whole word only, so `MAINTENANCE` or `CHAIR` don't match), `ARTIFICIAL INTELLIGENCE`, `DEEP LEARNING`, `NLP`, `COMPUTER VISION`, `LLM`
3. Data Engineer: `DATA ENGINEER`, `BIG DATA`, `DATA ARCHITECT`, `DATA WAREHOUS`, `ETL`, `DATA PLATFORM`, `DATA PIPELINE`, `ANALYTICS ENGINEER`, `DATA INTEGRATION`, `DATA DEVELOPER`, `DATA MODELER`
4. Statistician/Quant: `STATISTIC`, `BIOSTATISTIC`, `QUANT`, `QUANTITATIVE`, `ECONOMETRIC`, `OPERATIONS RESEARCH`
5. Data Analyst/BI: `DATA ANALY`, `ANALYTICS`, `BUSINESS INTELLIGENCE`, `BUSINESS INTEL`, `BI`, `POWER BI`, `TABLEAU`, `REPORTING ANALYST`, `INSIGHTS ANALYST`, `DATA VISUALI`

Inside **core** codes only, `BUSINESS ANALYST`, `BUSINESS SYSTEMS ANALYST` and `REPORTING` also map to Data Analyst/BI.

**Exclusions** (a title rule never counts for these): `DATA ENTRY`, `DATABASE ADMINISTRATOR` / `DBA`, `DATA CENTER`,
`ANALYTICAL CHEMIST / SCIENTIST / DEVELOPMENT / LAB` (lab chemistry), `CLINICAL DATA MANAGER/COORDINATOR/ASSOCIATE`.

`role_match_reason` records why each row was matched: `soc`, `soc+title` or `title`.

**Generic titles are flagged, not dropped.** Some rows count only because of their core SOC code while the title says nothing about data (Associate, Vice President, Product Manager, Business Manager, Software Engineer ...). The employer still told DOL this is a data-science/statistics job, so the row stays, but `title_is_generic = TRUE` (patterns in `GENERIC_TITLE_PATTERNS`) lets the dashboard hide it on title-level pages.

In [13]:
df = cl.classify_roles(df)
steps["8. classified (all rows)"] = len(df)
display(df["role_family"].value_counts())
pd.crosstab(df["role_family"], df["role_match_reason"], margins=True)

role_family
Other                 382288
Data Analyst/BI        15096
Data Scientist         11969
Data Engineer          10642
Statistician/Quant     10087
ML/AI Engineer          7414
Name: count, dtype: int64[pyarrow]

role_match_reason,none,soc,soc+title,title,All
role_family,,,,,
Data Analyst/BI,0,2791,10102,2203,15096
Data Engineer,0,174,4347,6121,10642
Data Scientist,0,3320,8135,514,11969
ML/AI Engineer,0,0,2268,5146,7414
Other,382288,0,0,0,382288
Statistician/Quant,0,5642,3984,461,10087
All,382288,11927,28836,14445,437496


In [14]:
# Generic-title flag: rows kept only by their SOC code whose title has no data meaning
g = df[df["title_is_generic"]]
print(f"flagged generic: {len(g):,} of {int(df['is_data_role'].sum()):,} data-role rows")
g["job_title_base"].value_counts().head(15)

flagged generic: 3,649 of 55,208 data-role rows


job_title_base
CONSULTANT                 400
ASSOCIATE                  383
ANALYST                    321
PRODUCT MANAGER            261
MANAGER                    249
SENIOR ASSOCIATE           232
VICE PRESIDENT             159
SOFTWARE ENGINEER          153
BUSINESS MANAGER           135
FINANCIAL ANALYST           69
DIRECTOR                    68
DEPLOYMENT STRATEGIST       66
PROGRAM MANAGER             57
TRADER                      55
TECH BUSINESS DEVELOPER     52
Name: count, dtype: int64[pyarrow]

In [15]:
# Where do title-only matches come from? (the SOC codes they were filed under)
t = df[df["role_match_reason"] == "title"]
pd.crosstab(t["soc_6"], t["role_family"], margins=True).sort_values("All", ascending=False).head(12)

role_family,Data Analyst/BI,Data Engineer,Data Scientist,ML/AI Engineer,Statistician/Quant,All
soc_6,,,,,,
All,2203,6121,514,5146,461,14445
15-1252,583,3250,83,3420,132,7468
15-1243,123,1342,11,35,4,1515
15-1299,199,344,60,316,4,923
15-1221,5,16,59,764,12,856
11-3021,152,228,101,225,13,719
15-1242,136,515,14,6,1,672
15-1211,216,108,5,51,1,381
15-1251,58,162,4,91,27,342


In [16]:
# 15-1252 Software Developers rows that the title rules pulled in, by family
sd = df[(df["soc_6"] == "15-1252") & df["is_data_role"]]
print("15-1252 rows classified as data roles:", len(sd))
sd["role_family"].value_counts()

15-1252 rows classified as data roles: 7468


role_family
ML/AI Engineer        3420
Data Engineer         3250
Data Analyst/BI        583
Statistician/Quant     132
Data Scientist          83
Name: count, dtype: int64[pyarrow]

### Sample of matched titles (one table per family)
Read these lists. If a title looks wrong, change the rule in `src/clean_lca.py` and re-run.

In [17]:
for fam in cl.ROLE_FAMILIES:
    s = df.loc[df["role_family"] == fam, "job_title_clean"].value_counts()
    print(f"\n=== {fam}: {s.sum():,} rows, {len(s):,} distinct titles ===")
    print(s.head(15).to_string())


=== Data Scientist: 11,969 rows, 2,652 distinct titles ===
job_title_clean
DATA SCIENTIST                      3078
SENIOR DATA SCIENTIST               1038
DATA SCIENTIST II                    319
DATA SCIENTIST III                   193
APPLIED SCIENCES                     172
DATA SCIENCE                         172
STAFF DATA SCIENTIST                 164
LEAD DATA SCIENTIST                  158
DATA SCIENTIST I                     122
SENIOR MANAGER DATA SCIENCE          114
PRINCIPAL DATA SCIENTIST             106
MANAGER DATA SCIENCE                 101
BUSINESS MANAGER                     101
DATA SCIENCE ENGINEER                 81
PRINCIPAL ASSOCIATE DATA SCIENCE      78

=== ML/AI Engineer: 7,414 rows, 2,270 distinct titles ===
job_title_clean
MACHINE LEARNING ENGINEER                    846
AI ENGINEER                                  391
SENIOR MACHINE LEARNING ENGINEER             377
MACHINE LEARNING                             309
SOFTWARE ENGINEER MACHINE LEARNING    

In [18]:
# Core SOC rows where the TITLE did not match (family taken from the SOC code). Check these.
soc_only = df[df["role_match_reason"] == "soc"]
soc_only.groupby("role_family")["job_title_clean"].value_counts().groupby(level=0).head(8)

role_family         job_title_clean               
Data Analyst/BI     PRODUCT MANAGER                    61
                    TECH BUSINESS DEVELOPER III        39
                    DEPLOYMENT STRATEGIST              34
                    SENIOR PRODUCT MANAGER             31
                    PRODUCT MANAGEMENT                 30
                    ASSOCIATE                          29
                    TECHNICAL PROGRAM MANAGEMENT       29
                    SENIOR CONSULTANT                  26
Data Engineer       SOFTWARE ENGINEER                  14
                    SENIOR SOFTWARE ENGINEER            5
                    LEAD DATA MIGRATION CONSULTANT      4
                    SOFTWARE ENGINEERING ADVISOR        4
                    DATABRICKS CONSULTANT               4
                    SENIOR CONSULTANT                   3
                    SENIOR TECHNICAL CONSULTANT         3
                    ENGINEERING MANAGER                 3
Data Scientist      A

### Sample of excluded titles
These titles hit a data keyword but were excluded on purpose:

In [19]:
df.loc[df["title_excluded"] & df["title_family"].notna(), "job_title_clean"].value_counts().head(20)

job_title_clean
DATA ANALYST DATA CENTER OPERATIONS                          1
SENIOR EPIC ANALYTIC DBA ETL ADMINISTRATOR                   1
DATA SCIENTIST POSTGRES DBA                                  1
MANAGER DATABASE ADMINISTRATION CONFIGURATION / ANALYTICS    1
BUSINESS INTELLIGENCE EXPERT DATA CENTER                     1
DATABASE ADMINISTRATOR BUSINESS ANALYTICS                    1
ENTERPRISE DATABASE ADMINISTRATOR MID BIG DATA               1
DATA ANALYST CLINICAL DATA MANAGER                           1
DATA WAREHOUSE DBA                                           1
DATABASE ADMINISTRATOR / DATA ENGINEER                       1
DATABASE ADMINISTRATOR DATA ANALYST                          1
Name: count, dtype: int64[pyarrow]

### Data-looking titles left out (Rule 3)
These titles match a keyword but sit under SOC codes outside the gated list, such as engineering, teaching or
accounting codes. They're left out on purpose. Check this list for anything you think should count.

In [20]:
left_out = df[(df["role_family"] == "Other") & df["title_family"].notna() & ~df["title_excluded"]]
print("rows:", len(left_out))
display(left_out["soc_6"].value_counts().head(10))
left_out["job_title_clean"].value_counts().head(20)

rows: 1345


soc_6
17-2199    138
13-1082    100
13-2054     91
13-1081     85
17-2112     77
25-1022     55
43-9111     48
17-2071     47
41-9031     47
25-1011     39
Name: count, dtype: int64[pyarrow]

job_title_clean
STRATEGIC OPERATIONS LEAD AI DATA OPERATIONS    51
DATA ANALYST                                    26
MACHINE LEARNING ENGINEER                       19
AI HARDWARE ENGINEER                            19
SENIOR MACHINE LEARNING ENGINEER                13
DATA SCIENTIST                                  13
STATISTICAL ASSISTANT                           12
ASSISTANT PROFESSOR OF BUSINESS ANALYTICS       11
COMPUTER VISION ENGINEER                         9
DATA ENGINEER                                    9
RESEARCH DATA ANALYST                            8
ASSISTANT PROFESSOR OF STATISTICS                8
MACHINE LEARNING                                 8
STAFF AI HARDWARE ENGINEER                       8
SENIOR DATA SCIENTIST                            6
AI AGENT ENGINEER                                6
SENIOR DATA ANALYST                              6
SUPPLY CHAIN DATA ANALYST                        6
OPERATIONS RESEARCH SCIENTIST II                 5
ASSISTANT PROFE

## Step 9 · Keep data roles and save

`to_output()` keeps the rows where `is_data_role` is True and puts columns in a fixed order with lowercase names,
ready for PostgreSQL. **All statuses and visa classes are kept.** Filter with `is_certified_h1b` in SQL or Power BI,
so you can still show withdrawals and denials if you want.

In [21]:
out = cl.to_output(df)
steps["9. data roles (output)"] = len(out)
steps["   of which certified H-1B"] = int(out["is_certified_h1b"].sum())
pd.Series(steps, name="rows").to_frame()

,rows
1. raw rows,437496
2. after column selection,437496
3. after status/dates,437496
4. after SOC normalization,437496
5. after wage annualization,437496
6. after employer standardization,437496
7. after title normalization,437496
8. classified (all rows),437496
9. data roles (output),55208
of which certified H-1B,50384


## Sanity checks on the output

In [22]:
assert out["case_number"].is_unique
assert out["role_family"].isin(cl.ROLE_FAMILIES).all()
assert out["soc_valid"].all()
h = out[out["is_certified_h1b"]]
print("certified H-1B data-role rows:", len(h))
display(h["role_family"].value_counts())

certified H-1B data-role rows: 50384


role_family
Data Analyst/BI       13792
Data Scientist        11008
Data Engineer          9741
Statistician/Quant     9139
ML/AI Engineer         6704
Name: count, dtype: int64[pyarrow]

In [23]:
# Annual offered wage by role family (certified H-1B, flagged wages removed)
hw = h[h["wage_flag"].isin(["ok", "unit_assumed_year"])]
hw.groupby("role_family")["wage_annual_from"].describe(percentiles=[.25, .5, .75]).round(0)

,count,mean,std,min,25%,50%,75%,max
role_family,,,,,,,,
Data Analyst/BI,13792.0,120322.0,38367.0,39915.0,93800.0,112923.0,141773.0,515000.0
Data Engineer,9741.0,134855.0,41288.0,51314.0,106000.0,128669.0,156000.0,728000.0
Data Scientist,11008.0,144448.0,49390.0,42000.0,109491.0,138767.0,172170.0,731000.0
ML/AI Engineer,6703.0,179163.0,68306.0,45000.0,131997.0,171600.0,212659.0,832000.0
Statistician/Quant,9139.0,136868.0,53840.0,44554.0,96073.0,126860.0,166540.0,500000.0


In [24]:
# Top 15 employers (certified H-1B data-role filings)
h["employer_name_std"].value_counts().head(15)

employer_name_std
AMAZON COM SERVICES                         1733
META PLATFORMS                               827
JPMORGAN CHASE                               671
WAL MART ASSOCIATES                          664
APPLE                                        560
MICROSOFT                                    555
CAPITAL ONE NATIONAL ASSOCIATION             506
GOOGLE                                       478
CAPITAL ONE SERVICES                         396
ADOBE                                        339
LINKEDIN                                     319
TT COMMERCE AND GLOBAL SERVICES              293
GOLDMAN SACHS                                282
PRICEWATERHOUSECOOPERS ADVISORY SERVICES     280
DELOITTE CONSULTING                          275
Name: count, dtype: int64[pyarrow]

In [25]:
# Top worksite states, and wage levels
display(h["worksite_state"].value_counts().head(10))
h["pw_wage_level"].value_counts()

worksite_state
CA    9670
TX    7382
NY    6097
WA    3447
NJ    2761
IL    2372
MA    2030
NC    1808
GA    1677
VA    1559
Name: count, dtype: int64[pyarrow]

pw_wage_level
II              18400
III             10035
IV               9272
I                8148
Not reported     4529
Name: count, dtype: int64[pyarrow]

In [26]:
paths = cl.save_outputs(out, OUT_DIR, "lca_roles_fy2026q3")
for k, p in paths.items():
    print(k, p.relative_to(REPO), f"{p.stat().st_size/1e6:.1f} MB")
check = pd.read_parquet(paths["parquet"])
assert len(check) == len(out)
print("re-read OK:", check.shape)

parquet data/processed/lca_roles_fy2026q3.parquet 4.2 MB
csv data/processed/lca_roles_fy2026q3.csv 28.2 MB
re-read OK: (55208, 57)


## Notes for later quarters
* DOL's quarterly files for one fiscal year are **cumulative** (the Q3 file covers Oct 1 to Jun 30). When you stack
  several files, use `cl.combine_quarters()`. It drops duplicate `case_number` values and keeps the newest file's row.
* Older files can have slightly different column names. Add them to `COLUMN_RENAMES` in `src/clean_lca.py`.